## Example 1: Render `git describe` pieces as a PEP 440 version

In [ ]:
import versioneer

# Example 1: render the pieces of `git describe` as a PEP 440 version.
# Here: 3 commits after tag 1.2, at commit abc1234, with a clean working tree.
pieces = {
    "closest-tag": "1.2",
    "distance": 3,
    "short": "abc1234",
    "long": "abc1234" + "0" * 33,
    "dirty": False,
    "error": None,
    "branch": "main",
    "date": None,
}
result = versioneer.render(pieces, "pep440")
print(result)

assert result["version"] == "1.2+3.gabc1234", result
assert result["full-revisionid"] == pieces["long"]
assert result["dirty"] is False
assert result["error"] is None

## Example 2: Exact tags, dirty trees and errors

In [ ]:
import versioneer

# Example 2: a commit exactly on the tag, a dirty working tree, and a failed `git describe`.
base = {
    "closest-tag": "1.2",
    "distance": 3,
    "short": "abc1234",
    "long": "abc1234" + "0" * 33,
    "dirty": False,
    "error": None,
    "branch": "main",
    "date": None,
}

on_tag = versioneer.render(dict(base, distance=0), "pep440")
dirty = versioneer.render(dict(base, dirty=True), "pep440")
failed = versioneer.render(dict(base, error="no tags found"), "pep440")
print(f"on the tag:    {on_tag['version']}")
print(f"dirty tree:    {dirty['version']}")
print(f"with an error: {failed['version']} ({failed['error']})")

assert on_tag["version"] == "1.2", on_tag
assert dirty["version"] == "1.2+3.gabc1234.dirty", dirty
assert dirty["dirty"] is True
assert failed["version"] == "unknown", failed
assert failed["error"] == "no tags found"

## Example 3: Version styles

In [ ]:
import versioneer

# Example 3: the same pieces rendered in each version style.
pieces = {
    "closest-tag": "1.2",
    "distance": 3,
    "short": "abc1234",
    "long": "abc1234" + "0" * 33,
    "dirty": False,
    "error": None,
    "branch": "main",
    "date": None,
}
expected = {
    "pep440": "1.2+3.gabc1234",
    "pep440-pre": "1.2.post0.dev3",
    "pep440-post": "1.2.post3+gabc1234",
    "pep440-old": "1.2.post3",
    "git-describe": "1.2-3-gabc1234",
}
for style, want in expected.items():
    got = versioneer.render(pieces, style)["version"]
    print(f"{style:13} -> {got}")
    assert got == want, (style, got)

try:
    versioneer.render(pieces, "no-such-style")
except ValueError as e:
    print(f"unknown style rejected: {e}")
else:
    raise AssertionError("render() accepted an unknown style")

## Example 4: Versions from git-archive keywords

In [ ]:
import os
import tempfile

import versioneer

# Example 4: in a `git archive` tarball, git fills the $Format$ placeholders in
# _version.py; versioneer reads the tag back from those keywords.
full = "abc1234" + "0" * 33
keywords = {"refnames": " (tag: v1.0)", "full": full, "date": None}
result = versioneer.git_versions_from_keywords(keywords, "v", False)
print(result)
assert result["version"] == "1.0", result
assert result["full-revisionid"] == full

# The same keywords, read from a _version.py file
with tempfile.TemporaryDirectory() as tmp:
    version_file = os.path.join(tmp, "_version.py")
    with open(version_file, "w") as f:
        f.write('git_refnames = " (tag: v3.1)"\n')
        f.write(f'git_full = "{full}"\n')
        f.write('git_date = "2024-01-02 03:04:05 +0000"\n')
    from_file = versioneer.git_get_keywords(version_file)
print(from_file)
result = versioneer.git_versions_from_keywords(from_file, "v", False)
print(result)
assert result["version"] == "3.1", result
assert result["date"] == "2024-01-02T03:04:05+0000", result

# Placeholders that git did not fill mean this is not a git-archive tarball
try:
    versioneer.git_versions_from_keywords({"refnames": "$Format:%d$", "full": full, "date": None}, "v", False)
except versioneer.NotThisMethod as e:
    print(f"NotThisMethod: {e}")
else:
    raise AssertionError("unfilled keywords were accepted")

## Example 5: Version from the parent directory name

In [ ]:
import os
import tempfile

import versioneer

# Example 5: an unpacked sdist named <prefix><version> gives its version through the directory name.
with tempfile.TemporaryDirectory() as tmp:
    root = os.path.join(tmp, "myproj-2.3")
    os.makedirs(root)
    result = versioneer.versions_from_parentdir("myproj-", root, False)
    print(result)
    assert result["version"] == "2.3", result
    assert result["full-revisionid"] is None

    try:
        versioneer.versions_from_parentdir("otherproj-", root, False)
    except versioneer.NotThisMethod as e:
        print(f"NotThisMethod: {e}")
    else:
        raise AssertionError("a directory without the prefix was accepted")